# 01 | Validate daily demand

Requires an existing Delta table `workspace.default.restaurant_daily_demand` with `business_day`, `orders`, and `net_sales`. The original POS-to-daily aggregation happened outside this notebook. Do not publish proprietary POS records.

In [ ]:
spark.sql("SELECT current_catalog(), current_schema()").show()

In [ ]:
from pyspark.sql import functions as F

TABLE_NAME = "workspace.default.restaurant_daily_demand"

df = spark.table(TABLE_NAME)

display(df.limit(10))

print("Total rows:", df.count())
print("Columns:", df.columns)

In [ ]:
from pyspark.sql import functions as F

daily = (
    df
    .withColumn(
        "business_day",
        F.to_date("business_day")
    )
    .withColumn(
        "orders",
        F.col("orders").cast("integer")
    )
    .withColumn(
        "net_sales",
        F.col("net_sales").cast("double")
    )
)

daily.select(
    F.count("*").alias("total_days"),
    F.min("business_day").alias("first_date"),
    F.max("business_day").alias("last_date"),
    F.avg("orders").alias("average_orders"),
    F.min("orders").alias("minimum_orders"),
    F.max("orders").alias("maximum_orders"),
    F.sum(
        F.when(
            F.col("business_day").isNull()
            | F.col("orders").isNull()
            | (F.col("orders") < 0),
            1
        ).otherwise(0)
    ).alias("invalid_rows"),
    (
        F.count("*") -
        F.countDistinct("business_day")
    ).alias("duplicate_dates")
).show()

In [ ]:
%sql

SELECT
    DAYOFWEEK(business_day) AS weekday_number,
    ROUND(AVG(orders), 2) AS avg_daily_orders,
    COUNT(*) AS recorded_days
FROM workspace.default.restaurant_daily_demand
GROUP BY DAYOFWEEK(business_day)
ORDER BY weekday_number;

In [ ]:
%sql

WITH calendar AS (
    SELECT EXPLODE(
        SEQUENCE(
            DATE('2024-03-29'),
            DATE('2026-10-03'),
            INTERVAL 1 DAY
        )
    ) AS business_day
)

SELECT
    DAYOFWEEK(c.business_day) AS weekday_number,
    COUNT(*) AS total_calendar_days,
    COUNT(d.business_day) AS days_with_orders,
    COUNT(*) - COUNT(d.business_day) AS missing_days
FROM calendar c
LEFT JOIN workspace.default.restaurant_daily_demand d
    ON c.business_day = d.business_day
GROUP BY DAYOFWEEK(c.business_day)
ORDER BY weekday_number;

In [ ]:
%sql

WITH calendar AS (
    SELECT EXPLODE(
        SEQUENCE(
            DATE('2024-03-29'),
            DATE('2026-10-03'),
            INTERVAL 1 DAY
        )
    ) AS business_day
)

SELECT
    c.business_day,
    DATE_FORMAT(c.business_day, 'EEEE') AS weekday
FROM calendar c
LEFT JOIN workspace.default.restaurant_daily_demand d
    ON c.business_day = d.business_day
WHERE d.business_day IS NULL
  AND DAYOFWEEK(c.business_day) <> 2
ORDER BY c.business_day;